# Channel-flow dataset analysis

Per-split (train / val / test) dataset analysis for the A1 channel flow.

The goal is to **compare splits against each other** to make sure train and test
are drawn from the same distribution — not to do a full physical audit.

**Part 1 — Statistical** (per component, per split):
  - mean and std of the velocity field
  - per-sample (per-snapshot) mean/std over time, to spot drift
  - train-vs-test comparison (delta, ratio) for each component

**Part 2 — Physical** (per split):
  - time-mean velocity profile along y
  - Friction Reynolds number over time
  - mean divergence over time

The dataset is passed as a parameter: a **RAM-backed** array (if it fits) or a
**dask-backed** lazy array (if it doesn't). The statistical code is identical
for both; only the array construction differs.

In [ ]:
import numpy as np
import zarr
import s3fs
import dask.array as da
import matplotlib.pyplot as plt
from tqdm.dask import TqdmCallback

# ---- config ----
S3_BUCKET  = "simulations"
STORE_PATH = "flow_export/A1"
ENDPOINT   = "http://localhost:8333"

Lx, Nx = 12.57, 128
Ly, Ny = 2.0, 129
Lz, Nz = 4.19, 128
nu = 1/4763

fs = s3fs.S3FileSystem(profile='default', client_kwargs={'endpoint_url': ENDPOINT})

# small channel grid (load directly)
def _charr(name):
    return np.asarray(zarr.open_array(
        store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/channel.zarr/{name}", s3=fs), mode="r"))
x, y, z = _charr('x'), _charr('y'), _charr('z')
print(f"grid: x[0]..x[-1] = {x[0]:.4f}..{x[-1]:.4f}  ({Nx} pts)")
print(f"      y[0]..y[-1] = {y[0]:.4f}..{y[-1]:.4f}  ({Ny} pts)")
print(f"      z[0]..z[-1] = {z[0]:.4f}..{z[-1]:.4f}  ({Nz} pts)")

## Time split (same as `full_2.ipynb`)

Val is contiguous near the start; train is strided (stride 4) in the middle;
test is the tail. A `spacing` gap separates the sets.

In [ ]:
def compute_splits(T, train_pc=80, val_pc=10, test_pc=10,
                   spacing=10, start_offset=60, train_stride=4):
    split_total = train_pc + val_pc + test_pc
    available_T = (T - spacing * 2 - start_offset)
    train_size = (available_T * train_pc) // split_total
    val_size   = (available_T * val_pc) // split_total
    test_size  = (available_T * test_pc) // split_total
    val_range   = range(start_offset, start_offset + val_size, 1)
    train_range = range(val_size + spacing + start_offset,
                        train_size + spacing + start_offset, train_stride)
    test_start  = train_size + spacing * 2 + start_offset
    test_range  = range(test_start, test_start + test_size, 1)
    return {"train": train_range, "val": val_range, "test": test_range}

T = 3001   # A1: (3001, 5, 128, 129, 128)
splits = compute_splits(T)
for k, r in splits.items():
    print(f"{k:5s}: n={len(r):5d}  first={r.start}  last={r.stop-1}  stride={r.step}")

## Dataset as a parameter

Pick **one** of the two backends. Both expose the same API:
  - `u` : velocity field `(T, 3, X, Y, Z)` (lazy dask array, or numpy in RAM)
  - `slice_split(name)` : return the sub-array restricted to a split's time range

The analysis code below does not care which backend you chose.

In [ ]:
USE_RAM = False   # True if the dataset fits in RAM; False for the lazy/dask path

class RamDataset:
    """Whole field loaded into RAM as a numpy array. Fast per-snapshot access."""
    def __init__(self):
        z = zarr.open_array(
            store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/data.zarr", s3=fs), mode="r")
        self._z = z
        self.u_all = z[:, :3]          # keep the reference; numpy slices are views
        self.shape = z.shape[:5]
        print(f"RAM dataset: {self.u_all.shape}  {z.dtype}  ~{z.nbytes/1024**3:.2f} GiB")

    def slice_split(self, name):
        r = splits[name]
        idx = np.array(list(r))
        return np.asarray(self.u_all[idx])     # (n, 3, X, Y, Z) numpy

class DaskDataset:
    """Lazy dask array over the S3 zarr. Only reductions are materialised."""
    CHUNK_TIME = 64
    def __init__(self):
        z = zarr.open_array(
            store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/data.zarr", s3=fs), mode="r")
        self._z = z
        self.u_all = da.from_array(z[:, :3], chunks=(self.CHUNK_TIME, 1, 128, 129, 128))
        self.shape = z.shape[:5]
        print(f"Dask dataset: {self.u_all.shape}  chunks time={self.CHUNK_TIME}")

    def slice_split(self, name):
        r = splits[name]
        # dask slicing by an integer range is cheap (no data moved yet)
        idx = np.array(list(r))
        return self.u_all[idx]

ds = RamDataset() if USE_RAM else DaskDataset()
print("active dataset:", type(ds).__name__)

# Part 1 — Statistical analysis (per component, per split)

For each split we compute:
- **global mean & std** of the velocity field (per component)
- **per-snapshot mean & std** over space (to see drift over time)

Then we compare splits: delta and ratio of mean/std between train and test.

In [ ]:
def split_stats(name, ds):
    """Return (mean_c, std_c, mean_per_snap, std_per_snap) for one split.

    mean_c, std_c   : (3,)         global per-component mean/std
    mean_per_snap   : (n, 3)       per-snapshot spatial mean (per component)
    std_per_snap    : (n, 3)       per-snapshot spatial std  (per component)
    """
    u = ds.slice_split(name)          # (n, 3, X, Y, Z)
    with TqdmCallback(desc=f"{name} mean/std"):
        mean_c = np.asarray(u.mean(axis=(0, 2, 3, 4)))   # (3,)
        std_c  = np.asarray(u.std(axis=(0, 2, 3, 4)))    # (3,)
    with TqdmCallback(desc=f"{name} per-snap"):
        mean_per_snap = np.asarray(u.mean(axis=(2, 3, 4)))  # (n, 3)
        std_per_snap  = np.asarray(u.std(axis=(2, 3, 4)))   # (n, 3)
    return mean_c, std_c, mean_per_snap, std_per_snap

stats = {name: split_stats(name, ds) for name in ("train", "val", "test")}
for name in ("train", "val", "test"):
    m, s, mps, sps = stats[name]
    print(f"{name:5s}  n={len(mps):4d}  "
          f"mean=[{m[0]:+.4f},{m[1]:+.4f},{m[2]:+.4f}]  "
          f"std=[{s[0]:.4f},{s[1]:.4f},{s[2]:.4f}]")

## Per-component mean/std comparison (train vs val vs test)

If the splits come from the same distribution, the per-component mean and std
should be close. Large gaps → the splits are not exchangeable.

In [ ]:
import pandas as pd

comp = ['ux', 'uy', 'uz']
rows = []
for c, lab in enumerate(comp):
    rows.append({
        "component": lab,
        "mean_train": stats['train'][0][c],
        "mean_val"  : stats['val'][0][c],
        "mean_test" : stats['test'][0][c],
        "std_train" : stats['train'][1][c],
        "std_val"   : stats['val'][1][c],
        "std_test"  : stats['test'][1][c],
        # relative differences (train vs test)
        "d_mean_test-train" : stats['test'][0][c] - stats['train'][0][c],
        "d_std_test-train"  : stats['test'][1][c] - stats['train'][1][c],
        "r_mean_test/train" : stats['test'][0][c] / stats['train'][0][c] if stats['train'][0][c] != 0 else np.nan,
        "r_std_test/train"  : stats['test'][1][c] / stats['train'][1][c],
    })
df = pd.DataFrame(rows).set_index("component")
df

In [ ]:
# grouped bar: mean per component across splits
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['train', 'val', 'test']
means = [stats[n][0] for n in labels]
stds  = [stats[n][1] for n in labels]

xpos = np.arange(len(comp))
w = 0.25
for i, lab in enumerate(labels):
    axes[0].bar(xpos + i*w, means[i], width=w, label=lab)
    axes[1].bar(xpos + i*w, stds[i],  width=w, label=lab)
axes[0].set_xticks(xpos + w); axes[0].set_xticklabels(comp)
axes[1].set_xticks(xpos + w); axes[1].set_xticklabels(comp)
axes[0].set_title('mean per component'); axes[0].legend()
axes[1].set_title('std  per component'); axes[1].legend()
plt.tight_layout(); plt.show()

## Per-snapshot drift (does the split drift over time?)

Plot the per-snapshot spatial mean/std for each component. Flat lines → the
split is stationary; a trend → the split drifts (train and test would not be
exchangeable if one drifts and the other doesn't).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex='all')
for c, lab in enumerate(comp):
    for i, name in enumerate(("train", "val", "test")):
        mps, sps = stats[name][2][:, c], stats[name][3][:, c]
        axes[0, c].plot(mps, label=name, lw=0.8)
        axes[1, c].plot(sps, label=name, lw=0.8)
    axes[0, c].set_title(f"{lab}: per-snapshot mean")
    axes[1, c].set_title(f"{lab}: per-snapshot std")
    axes[0, c].legend(fontsize=7)
    axes[1, c].legend(fontsize=7)
for ax in axes[-1]:
    ax.set_xlabel('snapshot index within split')
plt.suptitle('per-split drift (each split plotted against its own index)', y=1.02)
plt.tight_layout(); plt.show()

# Part 2 — Physical analysis (per split)

Only meaningful if you are training the **velocity** field directly. For each
split we compute:
- time-mean velocity profile along y
- Friction Reynolds number over time
- mean divergence over time

If the splits come from the same flow regime, these should look similar across
train / val / test.

In [ ]:
def profile_y(name, ds):
    u = ds.slice_split(name)
    with TqdmCallback(desc=f"{name} profile-y"):
        return np.asarray(u.mean(axis=(0, 2, 4)))   # (3, 129)

def friction_re(name, ds):
    u = ds.slice_split(name)
    with TqdmCallback(desc=f"{name} friction-Re"):
        u_mean = np.asarray(u[:, 0].mean(axis=(2, 4)))   # (n, 129) streamwise, mean over x,z
    du_dy = np.gradient(u_mean, y, axis=1)               # (n, 129)
    wall_grads = np.array([du_dy[:, 0], -du_dy[:, -1]])  # (2, n)
    u_tau = np.sqrt(nu * wall_grads.mean(axis=0))         # (n,)
    return u_tau / nu                                       # (n,) friction Re per snapshot

def mean_divergence(name, ds):
    """Mean of div·u per snapshot, derivatives via FFT (periodic x,z) + gradient y."""
    u = ds.slice_split(name)
    n = u.shape[0]
    divs = np.empty(n)
    for i in range(n):
        snap = np.asarray(u[i])                            # (3, X, Y, Z)
        def dfft(f, axis, h):
            nn = f.shape[axis]
            k = 2*np.pi*np.fft.fftfreq(nn, d=h)
            shape = [1]*f.ndim; shape[axis] = nn
            kb = np.broadcast_to(k.reshape(shape), f.shape)
            return np.fft.ifftn(1j*kb*np.fft.fftn(f, axes=(axis,)), axes=(axis,)).real
        dux = dfft(snap[0], 0, Lx/Nx)
        dvy = dfft(snap[1], 1, Ly/(Ny-1))
        dwz = dfft(snap[2], 2, Lz/Nz)
        divs[i] = float((dux + dvy + dwz).mean())
    return divs

phys = {name: {
    "profile": profile_y(name, ds),
    "re_friction": friction_re(name, ds),
    "div": mean_divergence(name, ds),
} for name in ("train", "val", "test")}
print("physical analysis done")

## Time-mean velocity profile along y (per split)

In [ ]:
plt.figure(figsize=(8, 5))
for name in ("train", "val", "test"):
    for c, lab in enumerate(comp):
        plt.plot(y, phys[name]["profile"][c], label=f"{lab}-{name}", lw=1.2)
plt.title('time-mean velocity profile along y (per split)')
plt.xlabel('y'); plt.legend(fontsize=7); plt.show()

## Friction Reynolds number over time (per split)

In [ ]:
plt.figure(figsize=(10, 4))
for name in ("train", "val", "test"):
    plt.plot(phys[name]["re_friction"], label=name, lw=1.0)
plt.title('Friction Re over time (per split)')
plt.xlabel('snapshot within split'); plt.ylabel('Re_\u03c4'); plt.legend(); plt.show()
for name in ("train", "val", "test"):
    re = phys[name]["re_friction"]
    print(f"{name:5s}  Re_\u03c4  mean={re.mean():.1f}  std={re.std():.1f}  min={re.min():.1f}  max={re.max():.1f}")

## Mean divergence over time (per split)

Should be ~0 for incompressible flow. Non-zero or split-dependent divergence
indicates a problem with the export or the split.

In [ ]:
plt.figure(figsize=(10, 4))
for name in ("train", "val", "test"):
    plt.plot(phys[name]["div"], label=name, lw=1.0)
plt.axhline(0, color='k', lw=0.5, ls='--')
plt.title('mean divergence over time (per split)')
plt.xlabel('snapshot within split'); plt.ylabel('mean div·u'); plt.legend(); plt.show()
for name in ("train", "val", "test"):
    d = phys[name]["div"]
    print(f"{name:5s}  div  mean={d.mean():+.3e}  max|div|={np.abs(d).max():.3e}")

In [ ]:
# no client to close (threads scheduler)